# Dự đoán test và đo thời gian · 5 lượt chính · Signa v1.1

Chấm test theo [cấu hình đã chốt](https://github.com/hana200804/DA/blob/fe8723b2c3776b9cb7928b412dd2691d7e6d93c0/ai/experiments/model-selection-v1.1.md#chốt-cấu-hình-trước-khi-đo-test): trọng số theo SHA-256, ảnh 640, ngưỡng vận hành chọn trên val. Không chọn lại ngưỡng trên test ([docs/04 §4](https://github.com/hana200804/DA/blob/fe8723b2c3776b9cb7928b412dd2691d7e6d93c0/docs/04-thiet-ke-ai.md#4-giao-thức-đánh-giá-và-chọn-mô-hình)).

**Mỗi nhóm chạy trong một runtime riêng.** YOLOv5 cài `requirements.txt` của repo gốc, kéo `ultralytics` mới hơn. YOLOv8 ghim `ultralytics==8.3.203`. DETR ghim `transformers==4.57.6`. Đổi nhóm thì chọn **Runtime → Disconnect and delete runtime** rồi mở lại.

| `SIGNA_GROUP` | Lượt | Môi trường |
| --- | --- | --- |
| `yolov8` | `yolov8n`, `yolov8s` | như `train_yolov8_main_v1.1.ipynb` |
| `yolov5` | `yolov5n`, `yolov5s` | repo YOLOv5 gốc `1e7cf74…`, như `train_yolov5_main_v1.1.ipynb`; ghim `ultralytics` theo `requirements.txt` của lượt train trên Drive, thiếu thì `8.4.201` |
| `detr` | `detr-r50` | `signa_ai.detr`, như `train_detr_main_v1.1.ipynb` |

1. Runtime → Change runtime type → **T4 GPU** (GPU khác cũng chạy được, `test-config.json` ghi tên GPU).
2. `signa-v1.1.zip` ở gốc My Drive. Trọng số ở `My Drive/Signa/experiments/<lượt>/weights/best.pt`, riêng DETR ở `My Drive/Signa/experiments/detr-r50-640-v1.1-main-s0/best.pt`.
3. Sửa ô mã đầu tiên: `SIGNA_GROUP` là `yolov8`, `yolov5` hoặc `detr`. Bỏ dấu `#` ở dòng `SIGNA_NOTEBOOK_COMMIT` và thay `<SHA>` bằng SHA trên permalink GitHub của notebook.
4. **Runtime → Run all.**

Với mỗi lượt, notebook làm theo thứ tự:

1. Kiểm SHA-256 trọng số với bảng chốt. Lệch thì dừng.
2. Xuất lại dự đoán **val** bằng đúng hàm notebook train dùng để xuất `predictions-val.json`, chấm bằng `signa_ai.evaluate` và so với `ai/experiments/<lượt>/metrics.json`. Nếu |Δ mAP@0.5:0.95| hoặc |Δ mAP@0.5| vượt 0,2 điểm phần trăm thì dừng, không chạy test.
3. Xuất dự đoán **test** (conf 0,001) và chấm với `--threshold` là ngưỡng đã chốt.
4. Đo thời gian đầu cuối trên 200 ảnh val, khởi động 10 ảnh, trên GPU rồi CPU của runtime. Đây **không phải** server tham chiếu (Q7 chưa chốt), số chỉ để tham khảo cho NFR-02.

Kết quả ở `My Drive/Signa/experiments/<lượt>/evaluation/`: `predictions-val-repro.json`, `metrics-val-repro.json`, `reproduce-val.json`, `predictions-test.json`, `metrics-test.json`, `test-config.json`, `latency.json`. Nhật ký ở `My Drive/Signa/experiments/predict-test-v1.1-<nhóm>.log`, đọc lại được khi Colab ẩn stderr.

10 ảnh khởi động là 10 ảnh đầu của chính 200 ảnh đo. Trên GPU, DETR chạy FP16 autocast, YOLO chạy FP32, như đường val của từng mô hình (trường `precision`). Lỗi ở ô 1–2 (cài gói, giải nén, kiểm GT) chỉ hiện trong ô Colab, chưa vào nhật ký Drive.

Khi chạy lại, notebook chỉ dùng hồ sơ khớp cấu hình khóa, cổng tái tạo val và SHA-256 của các tệp đầu ra. Phép đo thời gian còn phải khớp phần cứng, số luồng và tập ảnh đo. Hồ sơ dở dang, bị lệch hoặc thiếu SHA theo định dạng mới làm notebook dừng trước khi ghi kết quả. Để chạy mới, lưu riêng hồ sơ cũ trước. Kết quả lịch sử vẫn dùng để chấm lại từ dự đoán đã lưu.

In [ ]:
# Chọn một nhóm cho runtime này: yolov8, yolov5 hoặc detr. Đổi nhóm thì tạo runtime mới.
%env SIGNA_GROUP=yolov8
# %env SIGNA_NOTEBOOK_COMMIT=<SHA>

In [ ]:
# 1 · Cài môi trường của nhóm và lấy mã nguồn ghim commit.
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" not in sys.modules:
    raise RuntimeError("Notebook này chạy trên Colab có GPU (T4 là đủ).")

REPO_COMMIT = "fe8723b2c3776b9cb7928b412dd2691d7e6d93c0"
# Đặt SIGNA_NOTEBOOK_COMMIT theo SHA trên permalink GitHub của notebook đang chạy.
NOTEBOOK_COMMIT = os.environ.get("SIGNA_NOTEBOOK_COMMIT") or None
REPO = Path("/content/signa-source")
YOLOV5 = Path("/content/yolov5")
YOLOV5_COMMIT = "1e7cf74cb64d719b34f7315aa13eab9e32cde672"
TRANSFORMERS_VERSION = "4.57.6"
# requirements.txt của YOLOv5 1e7cf74 chỉ ghi ultralytics>=8.4.201, mà AutoShape gọi
# ultralytics.utils.ops. Ghim bản do freeze() của lượt train ghi trên Drive; thiếu tệp
# thì dùng 8.4.201, cận dưới đó và cũng là bản notebook xuất TFLite YOLOv5 dùng.
YOLOV5_ULTRALYTICS_FALLBACK = "8.4.201"

# Gói giống ô cài đặt của notebook train từng nhóm.
GROUPS = {
    "yolov8": {
        "runs": ["yolov8n-640-v1.1-main-s0", "yolov8s-640-v1.1-main-s0"],
        "packages": ["ultralytics==8.3.203", "albumentations==2.0.8", "pycocotools"],
    },
    "yolov5": {
        "runs": ["yolov5n-640-v1.1-main-s0", "yolov5s-640-v1.1-main-s0"],
        "packages": ["albumentations==2.0.8", "pycocotools"],
    },
    "detr": {
        "runs": ["detr-r50-640-v1.1-main-s0"],
        "packages": [
            "transformers==4.57.6", "timm==1.0.24", "albumentations==2.0.8",
            "pycocotools==2.0.11", "scipy==1.17.1",
        ],
    },
}
# Bảng "Chốt cấu hình trước khi đo test" trong ai/experiments/model-selection-v1.1.md.
LOCKED = {
    "yolov5n-640-v1.1-main-s0": {
        "weights": "weights/best.pt",
        "imgsz": 640,
        "threshold": 0.390178382396698,
        "weights_sha256": "1a41dfc18db0a00f2455b9159a902830de2b203d1b9c858a26a7d016be9a9936",
    },
    "yolov5s-640-v1.1-main-s0": {
        "weights": "weights/best.pt",
        "imgsz": 640,
        "threshold": 0.4176335334777832,
        "weights_sha256": "f47138cc1af5db74e912d0a86ca397eee69edf2f644d2f82383a68452c966a0f",
    },
    "yolov8n-640-v1.1-main-s0": {
        "weights": "weights/best.pt",
        "imgsz": 640,
        "threshold": 0.43184080719947815,
        "weights_sha256": "5cb3927b7e740ac944b09a569c4dc5547553cec8a5cdae635d8148a456f82a03",
    },
    "yolov8s-640-v1.1-main-s0": {
        "weights": "weights/best.pt",
        "imgsz": 640,
        "threshold": 0.5279324054718018,
        "weights_sha256": "49012dc0aa68865114f3a09545cab371978e5a52567cced52ebb050f49bf219d",
    },
    "detr-r50-640-v1.1-main-s0": {
        "weights": "best.pt",
        "imgsz": 640,
        "threshold": 0.706530917812261,
        "weights_sha256": "d0f90fc5d92fa406ea2e4d96003c4714e075ca30b41f48febd3f68e57a50a179",
    },
}
# SHA-256 của coco/val.json đã dùng để chấm val (val_gt_sha256 trong export-meta TFLite).
VAL_GT_SHA256 = "e3ccd276014eadd148030941675efeaf06db1c1ad8deddcd80747d40ec8b08e2"
TEST_GT_SHA256 = "38cad06f9b0c2960d4c4049f0ebd0de71f8bcefbef472022af10acc0031a9950"
EXPECTED = {"val": (1599, 2685), "test": (1700, 2664)}
EVAL_IOU = 0.7  # NMS của YOLO, như notebook train
MAX_DET = 300
REPRO_TOLERANCE = 0.002  # 0,2 điểm phần trăm trên thang mAP 0–1
LATENCY_IMAGES = 200
LATENCY_WARMUP = 10
DETR_BATCH = 16  # batch của lượt DETR (config.yaml), cũng là batch xuất val


def read_group(environ):
    group = environ.get("SIGNA_GROUP", "").strip().lower()
    if group not in GROUPS:
        raise ValueError(f"Đặt SIGNA_GROUP là một trong {', '.join(GROUPS)}, đang là {group!r}.")
    return group


def training_pin(requirements, package="ultralytics", fallback=YOLOV5_ULTRALYTICS_FALLBACK):
    """Phiên bản gói trong các requirements.txt của lượt train, kèm nguồn của phiên bản."""
    found = set()
    for text in requirements:
        for line in text.splitlines():
            name, separator, pinned = line.strip().partition("==")
            if separator and name.strip().lower() == package:
                found.add(pinned.strip())
    if len(found) > 1:
        raise RuntimeError(f"Các lượt train ghi nhiều bản {package}: {sorted(found)}")
    if found:
        return found.pop(), "requirements.txt của lượt train trên Drive"
    return fallback, "không có requirements.txt của lượt train, dùng cận dưới của YOLOv5 1e7cf74"


def sh(*args):
    subprocess.run([str(arg) for arg in args], check=True)


def checkout(url, destination, commit):
    if not (destination / ".git").exists():
        sh("git", "clone", "-q", url, destination)
    sh("git", "-C", destination, "fetch", "-q", "origin")
    sh("git", "-C", destination, "checkout", "-q", "--detach", commit)


GROUP = read_group(os.environ)
RUNS = GROUPS[GROUP]["runs"]
print("Nhóm", GROUP, "·", ", ".join(RUNS))

sh(sys.executable, "-m", "pip", "install", "-q", *GROUPS[GROUP]["packages"])
checkout("https://github.com/hana200804/DA.git", REPO, REPO_COMMIT)
sys.path.insert(0, str(REPO / "ai/src"))
os.environ["PYTHONPATH"] = str(REPO / "ai/src")  # cho subprocess signa_ai.evaluate
if GROUP == "yolov5":
    from google.colab import drive

    drive.mount("/content/drive")
    records = [
        Path("/content/drive/MyDrive/Signa/experiments") / run / "requirements.txt"
        for run in RUNS
    ]
    ULTRALYTICS_PIN, ULTRALYTICS_PIN_SOURCE = training_pin(
        [path.read_text() for path in records if path.is_file()]
    )
    print(f"ultralytics=={ULTRALYTICS_PIN} ({ULTRALYTICS_PIN_SOURCE})")
    checkout("https://github.com/ultralytics/yolov5.git", YOLOV5, YOLOV5_COMMIT)
    sh(
        sys.executable, "-m", "pip", "install", "-q", "-r", YOLOV5 / "requirements.txt",
        f"ultralytics=={ULTRALYTICS_PIN}",
    )
    # Repo gốc dùng anchor (docs/04 §2), không dùng yolov5u của gói ultralytics.
    sys.path.insert(0, str(YOLOV5))

In [ ]:
# 2 · Dữ liệu và GT COCO val/test từ signa-v1.1.zip.
import hashlib
import json
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import torch
from google.colab import drive

from signa_ai.dataset import read_yolo, relative_path
from signa_ai.detection import EVAL_CONFIDENCE, Box, Detection, to_coco_predictions
from signa_ai.labels import load_label_map

assert torch.cuda.is_available(), "Chọn GPU (T4 là đủ) trong Runtime → Change runtime type."
DEVICE = torch.device("cuda:0")
GPU = torch.cuda.get_device_name(DEVICE)
CPU_COUNT = os.cpu_count()
CPU_THREADS = min(4, CPU_COUNT or 1)  # giới hạn luồng PyTorch khi đo CPU
WORKERS = min(8, CPU_COUNT or 2)  # DataLoader DETR, như notebook train
print(f"GPU {GPU} · CPU {CPU_COUNT} · luồng đo CPU {CPU_THREADS}")

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
ARCHIVE = DRIVE / "signa-v1.1.zip"
DATASET = Path("/content/v1.1")
EXPERIMENTS = DRIVE / "Signa/experiments"
LOG_PATH = EXPERIMENTS / f"predict-test-v1.1-{GROUP}.log"

if not DATASET.is_dir():
    assert ARCHIVE.is_file(), f"Thiếu {ARCHIVE}"
    with TemporaryDirectory(dir=DATASET.parent) as staging, ZipFile(ARCHIVE) as archive:
        for member in archive.namelist():
            assert relative_path(member).parts[0] == DATASET.name, f"ZIP sai thư mục: {member}"
        archive.extractall(staging)
        (Path(staging) / DATASET.name).rename(DATASET)

LABELS = REPO / "ai/data/labels.yaml"
MANIFEST = DATASET / "manifest.csv"
label_map = load_label_map(LABELS)
NAMES = label_map.names
assert label_map.version == 1 and len(label_map) == 46
assert MANIFEST.read_bytes() == (REPO / "ai/data/manifests/v1.1.csv").read_bytes()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def check_gt(paths):
    digests = {}
    for split, expected in (("val", VAL_GT_SHA256), ("test", TEST_GT_SHA256)):
        digests[split] = sha256_file(paths[split])
        if digests[split] != expected:
            raise RuntimeError(f"coco/{split}.json: SHA-256 khác GT đã khóa {expected}. Dừng.")
    return digests


# GT là coco/<split>.json trong ZIP, cùng tệp notebook train dùng để chấm val.
GT_PATHS = {split: DATASET / f"coco/{split}.json" for split in EXPECTED}
GT_SHA256 = check_gt(GT_PATHS)

GROUND_TRUTH = {}
for split, (image_count, box_count) in EXPECTED.items():
    truth = json.loads(GT_PATHS[split].read_text())
    assert truth["info"]["label_map_version"] == label_map.version
    assert {c["id"]: c["name"] for c in truth["categories"]} == NAMES
    assert len(truth["images"]) == image_count and len(truth["annotations"]) == box_count
    boxes = 0
    for image in truth["images"]:
        rel = relative_path(image["file_name"])
        assert rel.parts[:2] == ("images", split) and (DATASET / rel).is_file(), str(rel)
        label = DATASET / "labels" / split / rel.with_suffix(".txt").name
        boxes += len(read_yolo(label, label_map))
    assert boxes == box_count
    GROUND_TRUTH[split] = truth
    print(split, image_count, "ảnh,", boxes, "box")


In [ ]:
# 3 · Ghi tệp, nhật ký, kiểm SHA, so sánh tái tạo và thống kê thời gian.
import math
import time
import traceback
from contextlib import contextmanager
from datetime import UTC, datetime
from importlib.metadata import PackageNotFoundError, version

LOG = None
PACKAGES = (
    "torch", "torchvision", "numpy", "pillow", "opencv-python", "opencv-python-headless",
    "pycocotools", "ultralytics", "transformers", "timm", "albumentations", "scipy",
)


def now():
    return datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")


def log(*parts):
    line = " ".join(str(part) for part in parts)
    print(line, flush=True)
    if LOG is not None:
        LOG.write(line + "\n")
        LOG.flush()


@contextmanager
def logged(path, title):
    """Ghi nhật ký kiểu export.log: mở đầu, nội dung, traceback nếu lỗi, kết thúc."""
    global LOG
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", buffering=1) as handle:
        LOG = handle
        status = "lỗi"
        try:
            log(f"\n--- {now()} · {title} ---")
            yield
            status = "xong"
        except BaseException:
            handle.write(traceback.format_exc())
            raise
        finally:
            log(f"--- {now()} · {status} ---")
            LOG = None


def atomic_write(path, write):
    staged = path.with_name(f".{path.name}.tmp")
    with staged.open("wb") as stream:
        write(stream)
        stream.flush()
        os.fsync(stream.fileno())
    staged.replace(path)


def write_text(path, content):
    atomic_write(path, lambda stream: stream.write(content.encode("utf-8")))


def write_json(path, data):
    write_text(path, json.dumps(data, ensure_ascii=False, allow_nan=False, indent=2) + "\n")


def check_weights(path, expected):
    """Trả SHA-256 của trọng số. Thiếu tệp hoặc lệch bảng chốt thì dừng."""
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu trọng số {path}")
    digest = sha256_file(path)
    if digest != expected:
        raise RuntimeError(f"{path}: SHA-256 {digest} khác bảng chốt {expected}. Dừng.")
    return digest


def reproduction_check(reproduced, reference, tolerance=REPRO_TOLERANCE):
    """So mAP val xuất lại với metrics.json đã commit. Sai số tính trên thang 0–1."""
    keys = ("map_50_95", "map_50")
    for name, metrics in (("tái tạo", reproduced), ("tham chiếu", reference)):
        for key in keys:
            value = metrics.get(key)
            if type(value) not in (int, float) or not math.isfinite(value):
                raise ValueError(f"metrics {name} thiếu {key} hữu hạn: {value!r}")
    delta = {key: reproduced[key] - reference[key] for key in keys}
    return {
        "reference": {key: reference[key] for key in keys},
        "reproduced": {key: reproduced[key] for key in keys},
        "delta": delta,
        "delta_points": {key: value * 100 for key, value in delta.items()},
        "tolerance": tolerance,
        "predictions": {
            "reference": reference.get("predictions"),
            "reproduced": reproduced.get("predictions"),
        },
        # 1e-12 bỏ qua sai số làm tròn khi Δ đúng bằng ngưỡng.
        "passed": all(abs(value) <= tolerance + 1e-12 for value in delta.values()),
    }


def read_report(path):
    try:
        report = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(report, dict):
            return report
    except (OSError, ValueError) as error:
        raise RuntimeError(
            f"{path}: không đọc được hồ sơ. Kiểm tra tay trước khi chạy lại."
        ) from error
    raise RuntimeError(f"{path}: cần đối tượng JSON. Kiểm tra tay trước khi chạy lại.")


def require_fields(report, expected, path):
    for field, value in expected.items():
        if field not in report or report[field] != value:
            raise RuntimeError(f"{path}: lệch hoặc thiếu {field}. Kiểm tra tay trước khi chạy lại.")


def test_identity(run):
    spec = LOCKED[run]
    export = {"conf": EVAL_CONFIDENCE, "imgsz": spec["imgsz"]}
    if GROUP == "detr":
        export["batch"] = DETR_BATCH
    else:
        export.update(iou=EVAL_IOU, max_det=MAX_DET)
    return {
        "run": run,
        "group": GROUP,
        "dataset_version": "v1.1",
        "label_map_version": label_map.version,
        "weights_sha256": spec["weights_sha256"],
        "test_gt_sha256": GT_SHA256["test"],
        "val_gt_sha256": GT_SHA256["val"],
        "manifest_sha256": sha256_file(MANIFEST),
        "export": export,
        "operating_threshold": spec["threshold"],
    }


def test_done(evaluation, identity):
    """Chỉ dùng lại bộ hồ sơ đầy đủ, khớp cấu hình đã khóa và cổng tái tạo val."""
    config_path = evaluation / "test-config.json"
    metrics_path = evaluation / "metrics-test.json"
    predictions_path = evaluation / "predictions-test.json"
    if not any(path.exists() for path in (config_path, metrics_path, predictions_path)):
        return False

    config = read_report(config_path)
    require_fields(config, identity, config_path)
    gate_path = evaluation / "reproduce-val.json"
    gate = read_report(gate_path)
    require_fields(gate, {
        "weights_sha256": identity["weights_sha256"],
        "val_gt_sha256": identity["val_gt_sha256"],
        "tolerance": REPRO_TOLERANCE,
    }, gate_path)
    delta = gate.get("delta")
    if (
        gate.get("passed") is not True
        or not isinstance(delta, dict)
        or any(
            type(delta.get(key)) not in (float, int)
            or not abs(delta[key]) <= REPRO_TOLERANCE + 1e-12
            for key in ("map_50", "map_50_95")
        )
    ):
        raise RuntimeError(
            f"{gate_path}: cổng tái tạo val chưa qua. Kiểm tra tay trước khi chạy lại."
        )
    require_fields(config, {
        "reproduction": {"delta": delta, "tolerance": REPRO_TOLERANCE, "passed": True},
    }, config_path)

    metrics = read_report(metrics_path)
    point = metrics.get("operating_point")
    if not isinstance(point, dict):
        raise RuntimeError(
            f"{metrics_path}: thiếu operating_point. Kiểm tra tay trước khi chạy lại."
        )
    require_fields(point, {"threshold": identity["operating_threshold"]}, metrics_path)
    for path, field in (
        (predictions_path, "predictions_test_sha256"), (metrics_path, "metrics_test_sha256")
    ):
        if not path.is_file():
            raise RuntimeError(f"{path}: thiếu tệp. Kiểm tra tay trước khi chạy lại.")
        require_fields(config, {field: sha256_file(path)}, path)
    return True


def latency_settings(identity):
    export = identity["export"]
    return {
        "images": LATENCY_IMAGES,
        "warmup": LATENCY_WARMUP,
        "image_ids": [image["id"] for image in GROUND_TRUTH["val"]["images"][:LATENCY_IMAGES]],
        **{key: export[key] for key in ("imgsz", "iou", "max_det") if key in export},
        "conf": identity["operating_threshold"],
    }


def latency_hardware(device):
    threads = {"torch_threads": CPU_THREADS, "cv2_threads": cv2.getNumThreads()}
    if device == "cuda":
        return {"device": GPU, "cuda": torch.version.cuda, **threads}
    return {"device": cpu_model(), "cpu_count": CPU_COUNT, **threads}


def latency_done(evaluation, identity):
    """Phép đo phải thuộc hồ sơ test đã kiểm và khớp môi trường đo hiện tại."""
    path = evaluation / "latency.json"
    if not path.exists():
        return False
    if not test_done(evaluation, identity):
        raise RuntimeError(f"{path}: thiếu hồ sơ test. Kiểm tra tay trước khi chạy lại.")

    config = read_report(evaluation / "test-config.json")
    report = read_report(path)
    require_fields(report, {
        **identity,
        **latency_settings(identity),
        "predictions_test_sha256": config["predictions_test_sha256"],
    }, path)
    for device in ("cuda", "cpu"):
        hardware = report.get(device)
        if not isinstance(hardware, dict):
            raise RuntimeError(f"{path}: thiếu phần đo {device}. Kiểm tra tay trước khi chạy lại.")
        require_fields(hardware, {
            **latency_hardware(device),
            "n": LATENCY_IMAGES,
            "precision": "fp16 autocast" if device == "cuda" and GROUP == "detr" else "fp32",
        }, f"{path} ({device})")
        for field in ("mean_ms", "p50_ms", "p95_ms", "min_ms", "max_ms"):
            value = hardware.get(field)
            if type(value) not in (int, float) or not math.isfinite(value) or value < 0:
                raise RuntimeError(f"{path} ({device}): {field} không hợp lệ. Dừng.")
    return True


def percentile(values, q):
    """Nội suy tuyến tính giữa hai hạng, như numpy.percentile mặc định."""
    ordered = sorted(values)
    if not ordered or not 0 <= q <= 100:
        raise ValueError("cần ít nhất một giá trị và q thuộc [0, 100]")
    position = (len(ordered) - 1) * q / 100
    lower = math.floor(position)
    upper = min(lower + 1, len(ordered) - 1)
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)


def latency_summary(samples):
    if not samples:
        raise ValueError("chưa có mẫu thời gian")
    return {
        "n": len(samples),
        "mean_ms": sum(samples) / len(samples),
        "p50_ms": percentile(samples, 50),
        "p95_ms": percentile(samples, 95),
        "min_ms": min(samples),
        "max_ms": max(samples),
    }


def time_pipeline(run, warmup, payloads, *, sync=None, clock=None):
    """Khởi động rồi đo từng ảnh, trả ms. Mỗi lần đo gồm giải mã bytes tới box ảnh gốc."""
    clock = clock or time.perf_counter
    for data in warmup:
        run(data)
    if sync is not None:
        sync()
    samples = []
    for data in payloads:
        start = clock()
        run(data)
        if sync is not None:
            sync()
        samples.append((clock() - start) * 1000)
    return samples


def package_versions(names=PACKAGES):
    versions = {}
    for name in names:
        try:
            versions[name] = version(name)
        except PackageNotFoundError:
            versions[name] = None
    return versions


def cpu_model():
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return None


def evaluate_cli(gt, predictions, out, threshold=None):
    """Chấm bằng CLI signa_ai.evaluate như notebook train, ghi stdout/stderr vào nhật ký."""
    staged = out.with_name(f".{out.name}.tmp")
    command = [
        sys.executable, "-m", "signa_ai.evaluate", "--gt", gt, "--pred", predictions,
        "--manifest", MANIFEST, "--labels", LABELS, "--out", staged,
    ]
    if threshold is not None:
        command += ["--threshold", repr(threshold)]
    result = subprocess.run(
        [str(arg) for arg in command], capture_output=True, text=True, env=os.environ
    )
    for stream in (result.stdout, result.stderr):
        if stream.strip():
            log(stream.rstrip())
    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    staged.replace(out)
    return json.loads(out.read_text(encoding="utf-8"))

In [ ]:
# 4 · Mô hình của nhóm. Hàm xuất COCO chép nguyên văn từ notebook train
# (tests/test_predict_test_notebook.py so AST với bản gốc).
import gc
from io import BytesIO

import cv2
import numpy as np
from PIL import Image

if GROUP == "yolov8":
    import ultralytics
    from ultralytics import YOLO

    assert ultralytics.__version__ == "8.3.203", "Restart runtime rồi chạy lại từ ô đầu."
elif GROUP == "yolov5":
    from models.common import AutoShape, DetectMultiBackend
    from utils.torch_utils import select_device

    assert version("ultralytics") == ULTRALYTICS_PIN, "Restart runtime rồi chạy lại từ ô đầu."
else:
    import transformers
    from torch.utils.data import DataLoader
    from tqdm.auto import tqdm

    from signa_ai.detr import DetrDataset, detr_detections, load_detr_checkpoint, predict_detr

    assert transformers.__version__ == TRANSFORMERS_VERSION, "Restart runtime rồi chạy lại."


# Từ train_yolov8_main_v1.1.ipynb (predict_with), chỉ đổi tên hàm.
def predict_with_yolov8(best_model):
    def predict_split(truth):
        predictions = []
        for image in truth["images"]:
            result = best_model.predict(
                str(DATASET / image["file_name"]), imgsz=640, conf=EVAL_CONFIDENCE,
                iou=0.7, max_det=300, device=0, verbose=False,
            )[0]
            assert result.orig_shape == (image["height"], image["width"])
            detections = [
                Detection(Box.from_xyxy(*xyxy), NAMES[int(cls)], float(score))
                for xyxy, cls, score in zip(
                    result.boxes.xyxy.cpu().tolist(),
                    result.boxes.cls.cpu().tolist(),
                    result.boxes.conf.cpu().tolist(),
                    strict=True,
                )
            ]
            predictions.extend(to_coco_predictions(image["id"], detections, label_map))
        return predictions

    return predict_split


# Từ train_yolov5_main_v1.1.ipynb (predict_with), chỉ đổi tên hàm.
def predict_with_yolov5(best_path):
    model = AutoShape(DetectMultiBackend(str(best_path), device=select_device("0")))
    model.conf, model.iou, model.max_det = EVAL_CONFIDENCE, 0.7, 300
    assert model.names == NAMES

    def predict_split(truth):
        predictions = []
        for image in truth["images"]:
            result = model(str(DATASET / image["file_name"]), size=640)
            assert result.ims[0].shape[:2] == (image["height"], image["width"])
            detections = [
                Detection(Box.from_xyxy(x1, y1, x2, y2), NAMES[int(cls)], float(score))
                for x1, y1, x2, y2, score, cls in result.xyxy[0].cpu().tolist()
            ]
            predictions.extend(to_coco_predictions(image["id"], detections, label_map))
        return predictions

    return predict_split


# Từ train_detr_main_v1.1.ipynb.
def collate(samples):
    pixels, masks, targets, image_ids, geometry = zip(*samples, strict=True)
    return (
        torch.from_numpy(np.stack(pixels)),
        torch.from_numpy(np.stack(masks)),
        [{key: torch.from_numpy(tensor) for key, tensor in target.items()} for target in targets],
        image_ids,
        geometry,
    )


@torch.inference_mode()
def predict_val(model, loader):
    model.eval()
    predictions = []
    for pixels, mask, _, image_ids, geometry in tqdm(loader, desc="Suy luận val"):
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(
                pixel_values=pixels.to(DEVICE, non_blocking=True),
                pixel_mask=mask.to(DEVICE, non_blocking=True),
            )
        batch_logits = outputs.logits.float().cpu().numpy()
        batch_boxes = outputs.pred_boxes.float().cpu().numpy()
        for image_id, shape, logits, boxes in zip(
            image_ids, geometry, batch_logits, batch_boxes, strict=True,
        ):
            detections = detr_detections(logits, boxes, shape, NAMES)
            predictions.extend(to_coco_predictions(image_id, detections, label_map))
    return predictions


def make_predictor(best_path):
    """Trả hàm split → dự đoán COCO conf 0,001, cùng đường xuất val của notebook train."""
    if GROUP == "yolov8":
        best_model = YOLO(str(best_path))
        assert best_model.names == NAMES
        predict_split = predict_with_yolov8(best_model)
        return lambda split: predict_split(GROUND_TRUTH[split])
    if GROUP == "yolov5":
        predict_split = predict_with_yolov5(best_path)
        return lambda split: predict_split(GROUND_TRUTH[split])

    # Kiến trúc từ configs/detr-r50-config.json, cùng tham số ghi đè như khi huấn luyện.
    model = load_detr_checkpoint(best_path, label_map, device=DEVICE)
    assert model.config.id2label == NAMES

    def predict_split(split):
        loader = DataLoader(
            DetrDataset(DATASET, split, label_map), batch_size=DETR_BATCH,
            num_workers=WORKERS, collate_fn=collate, pin_memory=True,
        )
        return predict_val(model, loader)

    return predict_split


def yolov8_pipeline(best_path, device, conf):
    model = YOLO(str(best_path))
    assert model.names == NAMES
    target = 0 if device == "cuda" else "cpu"

    def run(data):
        # Cùng cách ultralytics đọc ảnh từ đường dẫn (cv2.imdecode, BGR).
        image = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        result = model.predict(
            image, imgsz=640, conf=conf, iou=EVAL_IOU, max_det=MAX_DET,
            device=target, verbose=False,
        )[0]
        return [
            Detection(Box.from_xyxy(*xyxy), NAMES[int(cls)], float(score))
            for xyxy, cls, score in zip(
                result.boxes.xyxy.cpu().tolist(),
                result.boxes.cls.cpu().tolist(),
                result.boxes.conf.cpu().tolist(),
                strict=True,
            )
        ]

    return run, {"decode": "cv2.imdecode", "precision": "fp32"}


def yolov5_pipeline(best_path, device, conf):
    # select_device("cpu") đặt CUDA_VISIBLE_DEVICES=-1, nên CPU dùng torch.device trực tiếp.
    target = select_device("0") if device == "cuda" else torch.device("cpu")
    model = AutoShape(DetectMultiBackend(str(best_path), device=target))
    model.conf, model.iou, model.max_det = conf, EVAL_IOU, MAX_DET
    assert model.names == NAMES

    def run(data):
        # AutoShape đọc đường dẫn bằng PIL rồi exif_transpose, nên truyền ảnh PIL.
        with Image.open(BytesIO(data)) as image:
            result = model(image, size=640)
        return [
            Detection(Box.from_xyxy(x1, y1, x2, y2), NAMES[int(cls)], float(score))
            for x1, y1, x2, y2, score, cls in result.xyxy[0].cpu().tolist()
        ]

    return run, {"decode": "PIL.Image.open", "precision": "fp32"}


def detr_pipeline(best_path, device, conf):
    model = load_detr_checkpoint(best_path, label_map, device=device)

    def run(data):
        # predict_detr là hàm dịch vụ dùng: letterbox 640, softmax, chiếu box về ảnh gốc.
        with Image.open(BytesIO(data)) as image:
            return predict_detr(model, image, conf=conf)

    precision = "fp16 autocast" if device == "cuda" else "fp32"
    return run, {"decode": "PIL.Image.open", "precision": precision}


PIPELINES = {"yolov8": yolov8_pipeline, "yolov5": yolov5_pipeline, "detr": detr_pipeline}


def measure_latency(best_path, conf):
    images = GROUND_TRUTH["val"]["images"][:LATENCY_IMAGES]
    payloads = [(DATASET / image["file_name"]).read_bytes() for image in images]
    report = {}
    for device in ("cuda", "cpu"):
        torch.set_num_threads(CPU_THREADS)
        run, details = PIPELINES[GROUP](best_path, device, conf)
        # YOLOv8 có thể đổi số luồng khi khởi tạo predictor lần đầu.
        run(payloads[0])
        torch.set_num_threads(CPU_THREADS)
        hardware = latency_hardware(device)
        sync = torch.cuda.synchronize if device == "cuda" else None
        samples = time_pipeline(run, payloads[1:LATENCY_WARMUP], payloads, sync=sync)
        report[device] = {**hardware, **details, **latency_summary(samples)}
        summary = report[device]
        log(
            f"  {device}: p50 {summary['p50_ms']:.1f} ms · p95 {summary['p95_ms']:.1f} ms"
            f" · mean {summary['mean_ms']:.1f} ms"
        )
        del run
        gc.collect()
        torch.cuda.empty_cache()
    return report

In [ ]:
# 5 · Từng lượt: kiểm SHA → tái tạo val → test → đo thời gian.
def process(run):
    spec = LOCKED[run]
    run_dir = EXPERIMENTS / run
    evaluation = run_dir / "evaluation"
    evaluation.mkdir(parents=True, exist_ok=True)
    best_path = run_dir / spec["weights"]
    weights_sha = check_weights(best_path, spec["weights_sha256"])
    log(f"== {run}: trọng số khớp bảng chốt {weights_sha}")
    environment = {
        "repo_commit": REPO_COMMIT,
        "notebook_commit": NOTEBOOK_COMMIT,
        **(
            {
                "yolov5_commit": YOLOV5_COMMIT,
                "ultralytics_pin": ULTRALYTICS_PIN,
                "ultralytics_pin_source": ULTRALYTICS_PIN_SOURCE,
            }
            if GROUP == "yolov5"
            else {}
        ),
        "gpu": GPU,
        "cuda": torch.version.cuda,
        "python": sys.version.split()[0],
        "versions": package_versions(),
    }
    if NOTEBOOK_COMMIT is None:
        environment["notebook_commit_note"] = "Chưa cung cấp SHA qua SIGNA_NOTEBOOK_COMMIT."

    identity = test_identity(run)
    measured = latency_done(evaluation, identity)
    if measured or test_done(evaluation, identity):
        log("  hồ sơ test khớp cấu hình khóa và SHA-256, bỏ qua phần test")
    else:
        started = time.monotonic()
        predict_split = make_predictor(best_path)
        repro_path = evaluation / "predictions-val-repro.json"
        write_json(repro_path, predict_split("val"))
        reproduced = evaluate_cli(
            GT_PATHS["val"], repro_path, evaluation / "metrics-val-repro.json"
        )
        reference = json.loads((REPO / f"ai/experiments/{run}/metrics.json").read_text())
        check = reproduction_check(reproduced, reference)
        write_json(evaluation / "reproduce-val.json", {
            "weights_sha256": weights_sha, "val_gt_sha256": GT_SHA256["val"],
            "reference_path": f"ai/experiments/{run}/metrics.json", **check, **environment,
        })
        log(
            f"  tái tạo val: Δ mAP@0.5:0.95 {check['delta_points']['map_50_95']:+.4f} điểm,"
            f" Δ mAP@0.5 {check['delta_points']['map_50']:+.4f} điểm,"
            f" {check['predictions']['reproduced']} dự đoán"
            f" (gốc {check['predictions']['reference']})"
        )
        if not check["passed"]:
            raise RuntimeError(
                f"{run}: val xuất lại lệch quá {REPRO_TOLERANCE * 100:.1f} điểm. Không chạy test."
            )

        test_path = evaluation / "predictions-test.json"
        predictions = predict_split("test")
        write_json(test_path, predictions)
        metrics_path = evaluation / "metrics-test.json"
        metrics = evaluate_cli(
            GT_PATHS["test"], test_path, metrics_path, threshold=spec["threshold"],
        )
        # Ghi sau cùng: có test-config.json nghĩa là metrics-test.json đã xong.
        write_json(evaluation / "test-config.json", {
            **identity,
            "predictions_test_sha256": sha256_file(test_path),
            "metrics_test_sha256": sha256_file(metrics_path),
            "threshold_source": "ai/experiments/model-selection-v1.1.md, chọn trên val",
            "predictions": len(predictions),
            "reproduction": {key: check[key] for key in ("delta", "tolerance", "passed")},
            "scored_at": now(),
            "minutes": (time.monotonic() - started) / 60,
            **environment,
        })
        point = metrics["operating_point"]
        log(
            f"  test: mAP@0.5 {metrics['map_50']:.4f} · mAP@0.5:0.95 {metrics['map_50_95']:.4f}"
            f" · ngưỡng {point['threshold']} · P {point['precision']:.4f}"
            f" · R {point['recall']:.4f} · recall biển cấm {metrics['prohibitory_recall']:.4f}"
        )
        del predict_split
        gc.collect()
        torch.cuda.empty_cache()

    if measured:
        log("  hồ sơ thời gian khớp cấu hình và phần cứng, bỏ qua phần đo")
        return
    log(f"  đo thời gian: {LATENCY_IMAGES} ảnh val, khởi động {LATENCY_WARMUP} ảnh")
    latency = measure_latency(best_path, spec["threshold"])
    write_json(evaluation / "latency.json", {
        **identity,
        **latency_settings(identity),
        "predictions_test_sha256": read_report(
            evaluation / "test-config.json"
        )["predictions_test_sha256"],
        "note": (
            "Tham khảo cho NFR-02. Đo trên runtime Colab, không phải server tham chiếu "
            "(Q7 chưa chốt). Một tiến trình, từng ảnh một."
        ),
        "pipeline": (
            "bytes JPEG trong bộ nhớ → giải mã → tiền xử lý → suy luận → hậu xử lý "
            "→ box theo pixel ảnh gốc"
        ),
        "source": "ảnh đầu tiên của coco/val.json theo thứ tự trong tệp",
        "measured_at": now(),
        **environment,
        **latency,
    })


with logged(LOG_PATH, f"{GROUP} · repo {REPO_COMMIT} · notebook {NOTEBOOK_COMMIT}"):
    log(f"GPU {GPU} · CPU {cpu_model()} × {CPU_COUNT} · SHA-256 GT test {GT_SHA256['test']}")
    for run in RUNS:
        process(run)
print("Nhật ký:", LOG_PATH)

In [ ]:
# 6 · Tóm tắt các lượt của nhóm.
for run in RUNS:
    evaluation = EXPERIMENTS / run / "evaluation"
    metrics = json.loads((evaluation / "metrics-test.json").read_text())
    latency = json.loads((evaluation / "latency.json").read_text())
    point = metrics["operating_point"]
    print(
        f"{run}: mAP@0.5 {metrics['map_50']:.2%} · mAP@0.5:0.95 {metrics['map_50_95']:.2%}"
        f" · P {point['precision']:.2%} · R {point['recall']:.2%}"
        f" · recall biển cấm {metrics['prohibitory_recall']:.2%}"
        f" · p95 GPU {latency['cuda']['p95_ms']:.1f} ms · p95 CPU {latency['cpu']['p95_ms']:.1f} ms"
    )